# Файлы и форматы данных

Программа может читать данные из файла и сохранять результат, чтобы он не пропал после завершения работы. В этом уроке научимся работать с текстовыми файлами, путями, CSV и JSON.

> Ячейки создают учебные файлы в текущей папке блокнота. Запускай их по порядку.

## Карта урока

1. Открываем файл с помощью `with open(...)`
2. Читаем и записываем текст
3. Разбираемся с путями и кодировками
4. Читаем и записываем CSV
5. Читаем и записываем JSON
6. Практика: сохранить список дел


## 1. Открываем файл с помощью `with open(...)`

Чтобы читать или записывать файл, его сначала нужно открыть. Конструкция `with` закрывает файл автоматически, даже если во время работы возникнет ошибка.


![Открытие файла, чтение и автоматическое закрытие](assets/with_open.png)


In [ ]:
from pathlib import Path

notes_path = Path("lesson_notes.txt")

with open(notes_path, "w", encoding="utf-8") as file:
    file.write("Первая строка заметки.\n")
    file.write("Вторая строка заметки.\n")

Второй аргумент `"w"` означает режим записи. Если файла ещё нет, Python создаст его. Если файл уже есть, его прежнее содержимое будет заменено.

После блока `with` файл закрыт. Не нужно отдельно вызывать `file.close()`.

## 2. Читаем текст из файла

Режим `"r"` открывает файл для чтения. Метод `read()` читает всё содержимое.


In [ ]:
with open(notes_path, "r", encoding="utf-8") as file:
    text = file.read()

print(text)

Чтобы прочитать файл построчно, можно пройтись по нему циклом `for`. Такой способ удобен для больших файлов.


In [ ]:
with open(notes_path, "r", encoding="utf-8") as file:
    for line in file:
        print(line, end="")


В конце строки файла обычно уже есть `
`. `print(line)` добавит ещё один перенос. `print(line, end="")` сохраняет исходное содержимое без лишних пустых строк.

Если нужно убрать только перенос, используй `line.rstrip("\n")`. Вызов `rstrip()` без аргумента убирает также пробелы и табуляции справа, которые могут быть частью данных.

### Добавляем текст

Режим `"a"` добавляет данные в конец файла и создаёт файл, если его ещё нет.


In [ ]:
with open(notes_path, "a", encoding="utf-8") as file:
    file.write("Добавили ещё одну строку.\n")

with open(notes_path, "r", encoding="utf-8") as file:
    print(file.read())

Основные режимы открытия:

| Режим | Что делает |
|---|---|
| `"r"` | Читает существующий файл |
| `"w"` | Записывает с начала; старое содержимое заменяется |
| `"a"` | Добавляет в конец |
| `"x"` | Создаёт новый файл; выдаёт ошибку, если такой файл уже есть |

Для учебных файлов используй UTF-8 и при записи, и при чтении: `encoding="utf-8"`. Для чужого файла сначала выясни его кодировку.


### Содержимое файла и позиция чтения

Открытый файл помнит, до какого места мы прочитали. Второй `read()` без перемещения позиции вернёт пустую строку, если первый уже дошёл до конца. `seek(0)` вернёт позицию к началу.

В текстовом режиме `read(2)` читает два символа. Не вычисляй байтовые смещения кириллицы вручную: у UTF-8 число байтов и символов может различаться.


In [ ]:
from tempfile import TemporaryDirectory
from pathlib import Path

with TemporaryDirectory() as folder:
    path = Path(folder) / "sample.txt"
    path.write_text("кот", encoding="utf-8")
    with open(path, encoding="utf-8") as file:
        print(repr(file.read(2)))
        print(repr(file.read()))
        print(repr(file.read()))
        file.seek(0)
        print(file.read())
    print("Закрыт:", file.closed)


`repr()` показывает строку так, чтобы было видно пустой результат `''` и специальные символы.

**Задача.** Запиши две строки, прочитай файл дважды в одном `with`, затем повтори после `seek(0)`. Объясни разницу. Помни: `"w"` очищает существующее содержимое уже при открытии, даже если затем ничего не записать.


## 3. Пути к файлам

**Путь** указывает, где файл находится. Относительный путь считается от текущей рабочей папки программы. `Path.cwd()` показывает эту папку, а оператор `/` помогает соединять части пути.


In [ ]:
from pathlib import Path

current_folder = Path.cwd()
data_folder = current_folder / "lesson_data"
notes_file = data_folder / "notes.txt"

print("Текущая папка:", current_folder)
print("Путь к файлу:", notes_file)

Создадим отдельную папку для учебных данных. Параметр `parents=True` при необходимости создаёт и родительские папки, а `exist_ok=True` не вызывает ошибку, если папка уже существует.


In [ ]:
data_folder.mkdir(parents=True, exist_ok=True)

with open(notes_file, "w", encoding="utf-8") as file:
    file.write("Файл лежит в папке lesson_data.\n")

print(notes_file.exists())

Если читаемый файл не найден, Python поднимет `FileNotFoundError`. Обработай её, если отсутствие файла — ожидаемая ситуация.


In [ ]:
missing_file = data_folder / "not_created.txt"

try:
    with open(missing_file, "r", encoding="utf-8") as file:
        print(file.read())
except FileNotFoundError:
    print("Файл пока не создан.")

`pathlib` также умеет читать и записывать текст одной командой. Для первых упражнений используем `with open(...)`, чтобы привыкнуть к явным режимам и автоматическому закрытию.


In [ ]:
quick_note = data_folder / "quick_note.txt"
quick_note.write_text("Короткая запись через pathlib.\n", encoding="utf-8")
print(quick_note.read_text(encoding="utf-8"))

## 4. Формат CSV

CSV хранит таблицу: одна строка файла соответствует одной строке таблицы. Для работы с ним в Python есть встроенный модуль `csv`.

Передадим `newline=""` при открытии CSV-файла. Это помогает корректно записывать строки на разных операционных системах.


In [ ]:
import csv

students_path = data_folder / "students.csv"
students = [
    {"name": "Аня", "score": 5},
    {"name": "Борис", "score": 4},
    {"name": "Вера", "score": 5},
]

with open(students_path, "w", newline="", encoding="utf-8") as file:
    fieldnames = ["name", "score"]
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(students)

`DictReader` читает каждую строку как словарь, где ключи — это заголовки колонок.


In [ ]:
with open(students_path, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        print(row["name"], row["score"])

Значения, прочитанные из CSV, представлены строками. Если поле нужно использовать как число, преобразуй его явно.


In [ ]:
with open(students_path, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        score = int(row["score"])
        print(f"{row['name']}: {score + 1}")

CSV подходит для простых таблиц. Если данные вложены друг в друга или у разных записей разные поля, удобнее может оказаться JSON.

## 5. Формат JSON

JSON хранит структуры, похожие на словари, списки, строки, числа и логические значения Python. Для преобразования между Python и JSON используется модуль `json`.


In [ ]:
import json

profile = {
    "name": "Аня",
    "city": "Казань",
    "hobbies": ["шахматы", "робототехника"],
    "active": True,
}

profile_path = data_folder / "profile.json"
with open(profile_path, "w", encoding="utf-8") as file:
    json.dump(profile, file, ensure_ascii=False, indent=2)

Параметр `ensure_ascii=False` оставляет кириллицу читаемой в файле. `indent=2` добавляет отступы, чтобы JSON было удобно смотреть.

Прочитаем JSON обратно. Метод `json.load()` читает файл и превращает данные в словарь Python.


In [ ]:
with open(profile_path, "r", encoding="utf-8") as file:
    loaded_profile = json.load(file)

print(loaded_profile)
print(loaded_profile["hobbies"][0])

Запомни разницу:

- `json.dump(data, file)` записывает объект Python в открытый файл;
- `json.load(file)` читает объект Python из открытого файла.

Для строк есть `json.dumps(data)` и `json.loads(text)` — они не работают с файловым объектом.


In [ ]:
data_as_text = json.dumps(profile, ensure_ascii=False)
profile_copy = json.loads(data_as_text)

print(data_as_text)
print(profile_copy["city"])

JSON хранит данные, а не исполняемый код. В JSON нет прямого аналога кортежей и множеств: при сохранении используй поддерживаемые типы, например список.

## 6. Мини-проект: список дел

Сделаем список задач и сохраним его в JSON-файл. В каждой задаче будет текст и отметка, выполнена ли она.


In [ ]:
tasks_path = data_folder / "tasks.json"
tasks = [
    {"title": "Прочитать урок", "done": True},
    {"title": "Решить задачу", "done": False},
]

with open(tasks_path, "w", encoding="utf-8") as file:
    json.dump(tasks, file, ensure_ascii=False, indent=2)

Загрузим задачи и покажем их в удобном виде.


In [ ]:
with open(tasks_path, "r", encoding="utf-8") as file:
    saved_tasks = json.load(file)

for task in saved_tasks:
    mark = "✓" if task["done"] else " "
    print(f"[{mark}] {task['title']}")

### Практика

1. Добавь в список дел новую задачу.
2. Сохрани обновлённый список в `tasks.json`.
3. Прочитай файл ещё раз и выведи все задачи.
4. Добавь проверку на отсутствие файла при чтении.


In [ ]:
# Продолжи мини-проект здесь


## Короткое повторение

- Используй `with open(path, mode, encoding="utf-8") as file:` — файл закроется автоматически.
- `"r"` читает, `"w"` перезаписывает, `"a"` добавляет в конец.
- Для путей удобно применять `pathlib.Path`.
- Для CSV используй `csv.DictReader` и `csv.DictWriter`.
- Для JSON используй `json.load()` и `json.dump()`.
- Числа из CSV приходят строками, поэтому преобразуй их явно.
- Обрабатывай ожидаемую ошибку `FileNotFoundError`, если файла может не быть.

### Проверь себя

1. Чем режим `"a"` отличается от `"w"`?
2. Зачем указывать `encoding="utf-8"`?
3. Какой объект возвращает `json.load()` для JSON-объекта?
4. Какой параметр CSV-ридера даёт строкам имена колонок?


## Домашняя работа

Все условия и ячейки для решений находятся в этом блокноте. Обязательны задачи 1–3 и задача 4 на повторение. Остальные — по выбору вместе с преподавателем; не нужно выполнять весь банк за один вечер.

Для программ со стандартным вводом не добавляй приглашения к input и лишние подписи к ответу. Для функций таблицы описывают аргументы и возвращаемое значение; input нужен только там, где он указан. Для файлов и проектов таблицы описывают содержимое файлов, команды и ожидаемые результаты.

В таблицах примеров разные строки внутри ячейки означают разные строки данных. «Пустая строка (Enter)» и «Нет вывода» — поясняющие отметки, их не нужно вводить или печатать. Каждый пример проверяй отдельным запуском. Приведи ещё один собственный пример до написания кода.


### ДЗ-17.1. Библиотечный список · обязательная

**Сюжет.** Библиотекарь сохраняет выбранные книги между запусками.

**Условие.** Прочитайте три названия, запишите каждое отдельной строкой в books.txt, затем перечитайте файл и выведите строки с номерами.

**Формат данных**

| Входные данные | Выходные данные |
|---|---|
| Три строки стандартного ввода. Файл books.txt создаётся или перезаписывается в текущей рабочей папке. | Файл содержит три названия, каждое с переводом строки. В stdout — три строки «номер. название». |

**Ограничения.** Названия непустые до 80 символов; UTF-8; используйте with при записи и чтении.

**Примеры**

| Входные данные | Выходные данные |
|---|---|
| <pre>Дюна<br>Хоббит<br>Солярис</pre> | <pre>1. Дюна<br>2. Хоббит<br>3. Солярис</pre> |
| <pre>А<br>Б<br>В</pre> | <pre>1. А<br>2. Б<br>3. В</pre> |

**Источник.** Авторская задача курса.


### ДЗ-17.2. Новый участник списка · обязательная

**Сюжет.** К уже сохранённым книгам добавляют ещё одну, не стирая предыдущие.

**Условие.** Добавьте введённое название в books.txt через режим a. Затем выведите полное содержимое файла.

**Формат данных**

| Входные данные | Выходные данные |
|---|---|
| Входной books.txt и одна строка стандартного ввода с новым названием. | Дополненный файл и его текст в stdout. Не добавляйте лишние пустые строки. |

**Ограничения.** Файл существует; каждая прежняя строка, включая последнюю, заканчивается переводом строки; UTF-8.

**Примеры**

| Входные данные | Выходные данные |
|---|---|
| <pre>books.txt: Дюна<br>Новое название: Хоббит</pre> | <pre>Дюна<br>Хоббит</pre> |
| <pre>books.txt пустой<br>Новое название: Атлас</pre> | <pre>Атлас</pre> |

**Источник.** Авторская задача курса.


### ДЗ-17.3. Дорогие товары · обязательная

**Сюжет.** Склад хранит цены в CSV, а покупатель задаёт порог.

**Условие.** Прочитайте products.csv через DictReader и выведите товары строго дороже порога. Строки сохраняют порядок файла.

**Формат данных**

| Входные данные | Выходные данные |
|---|---|
| CSV с заголовком name,price; в stdin одно целое число — порог. | Строки «название цена» для совпадений; при отсутствии вывода нет. |

**Ограничения.** UTF-8, разделитель запятая; названия непустые без запятых; цены и порог целые от 0 до 10000. Файл может содержать только заголовок.

**Примеры**

| Входные данные | Выходные данные |
|---|---|
| <pre>products.csv:<br>name,price<br>чай,100<br>сок,150<br>Порог: 100</pre> | <pre>сок 150</pre> |
| <pre>products.csv:<br>name,price<br>Порог: 0</pre> | <em>Нет вывода</em> |

**Источник.** Авторская задача курса.


### ДЗ-17.4. Настройки игры · повторение

**Сюжет.** Игра хранит громкость и имя игрока в JSON.

**Условие.** Прочитайте settings.json, измените только volume на значение из stdin и сохраните JSON обратно.

**Формат данных**

| Входные данные | Выходные данные |
|---|---|
| JSON-объект с name и volume; затем целая громкость в stdin. | Обновлённый JSON-файл; stdout не требуется. Пробелы, отступы и экранирование кириллицы в JSON на оценку не влияют. |

**Ограничения.** Файл существует, UTF-8; name — строка, 0 ≤ volume ≤ 100.

**Примеры**

| Входные данные | Выходные данные |
|---|---|
| <pre>settings.json: {"name":"Аня","volume":20}<br>Ввод: 0</pre> | <pre>settings.json: {"name":"Аня","volume":0}</pre> |
| <pre>settings.json: {"name":"Ян","volume":0}<br>Ввод: 100</pre> | <pre>settings.json: {"name":"Ян","volume":100}</pre> |

**Источник.** Авторская задача курса.


### ДЗ-17.5. Список учеников в CSV · по выбору

**Сюжет.** Учитель переносит список записей в табличный файл.

**Условие.** save_students(records) записывает students.csv с заголовком name,score, затем читает файл через DictReader и возвращает восстановленный список словарей с целым score.

**Формат данных**

| Входные данные | Выходные данные |
|---|---|
| Аргумент — список словарей с name (строка) и score (целое). | CSV в UTF-8 и возвращаемый список, равный входному по содержимому. |

**Ограничения.** До 30 записей; 0 ≤ score ≤ 100; имена непустые, до 40 символов.

**Примеры**

| Входные данные | Выходные данные |
|---|---|
| <pre>save_students([{'name':'Аня','score':5}])</pre> | <pre>[{'name': 'Аня', 'score': 5}]</pre> |
| <pre>save_students([])</pre> | <pre>[]; файл содержит только заголовок name,score</pre> |

**Источник.** Авторская задача курса.


### ДЗ-17.6. Список дел между запусками · по выбору

**Сюжет.** Программа добавляет новую задачу к ранее сохранённым.

**Условие.** Прочитайте tasks.json как список строк, добавьте задачу из stdin и сохраните. Если файла нет, начните с пустого списка.

**Формат данных**

| Входные данные | Выходные данные |
|---|---|
| Необязательный tasks.json; одна непустая строка stdin — новое дело. | Обновлённый JSON-массив в файле; stdout — количество задач после добавления. |

**Ограничения.** Существующий JSON корректен; до 100 прежних строк. Пустой по содержимому файл не гарантируется корректным JSON и не входит в допустимый ввод.

**Примеры**

| Входные данные | Выходные данные |
|---|---|
| <pre>tasks.json: ["читать"]<br>Ввод: писать</pre> | <pre>2; файл: ["читать", "писать"]</pre> |
| <pre>Файла нет<br>Ввод: читать</pre> | <pre>1; файл: ["читать"]</pre> |

**Источник.** Авторская задача курса.


### ДЗ-17.7. Каталог по автору · по выбору

**Сюжет.** Читатель ищет книги конкретного автора в сохранённом каталоге.

**Условие.** В books.json лежит список объектов с title, author, year. Выведите названия книг, у которых author в точности совпадает с введённой строкой.

**Формат данных**

| Входные данные | Выходные данные |
|---|---|
| Существующий JSON-файл и строка автора в stdin. | Названия в порядке файла; если совпадений нет — «Не найдено». |

**Ограничения.** До 50 книг; названия/авторы непустые; регистр важен.

**Примеры**

| Входные данные | Выходные данные |
|---|---|
| <pre>books.json: [{"title":"Атлас","author":"Ли","year":2020}]<br>Ввод: Ли</pre> | <pre>Атлас</pre> |
| <pre>books.json: []<br>Ввод: Ли</pre> | <pre>Не найдено</pre> |

**Источник.** Авторская задача курса.


**Что сдать:** этот блокнот с решениями и сохранёнными результатами проверок. Если условие требует модули, файлы или тесты, приложи их к работе; не прикладывай виртуальное окружение. Для каждого решения добавь свой пример и коротко объясни выбор граничного случая.
